In [ ]:
import json
import pandas as pd
import numpy as np
import os
import torch
from transformers import AutoModelForCausalLM

import time

start_time = time.time()

def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
# ============================================================
# CONFIG
# ============================================================
path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
with open(path, "r") as f:
    dataset_days = json.load(f)

countries = ["Germany", "Ireland", "Portugal","Denmark"]
days = ["day1"]
#days = ["day1", "day2", "day3", "day4", "day5"]

LOOKBACK = 2880          # max supported context length from example
PRED_LEN = 96            # 96 steps ahead

# ============================================================
# LOAD TIMER MODEL
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = AutoModelForCausalLM.from_pretrained(
    "thuml/timer-base-84m",
    trust_remote_code=True
).to(device)

model.eval()

rmse_results = []


# ============================================================
# MAIN LOOP
# ============================================================
for country in countries:
    print("Processing country:", country)
    country_start_time = time.time()

    data_path = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    # --------------------------------------------------------
    # COUNTRY-SPECIFIC FEATURES
    # --------------------------------------------------------
    country_features = [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "precipitation",
        "direct_radiation"
    ]

    if country == "Denmark":
        country_features.append("price_eur_kwh")

    households = [col for col in df.columns if col not in country_features]

    # infer sampling frequency from dataframe index
    inferred_freq = pd.infer_freq(df.index)

    if inferred_freq is None:
        diffs = df.index.to_series().diff().dropna()
        step = diffs.median()
    else:
        step = pd.tseries.frequencies.to_offset(inferred_freq)

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:
            s_train = df.loc[df.index < cutoff, household].dropna()

            if len(s_train) < LOOKBACK:
                print(f"      Skipping {household}: not enough history ({len(s_train)} < {LOOKBACK})")
                continue

            context = s_train.iloc[-LOOKBACK:].astype("float32").values
            seqs = torch.tensor(context, dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                output = model.generate(seqs, max_new_tokens=PRED_LEN)

            y_pred = output.squeeze(0).detach().cpu().numpy()

            pred_index = pd.date_range(start=cutoff, periods=PRED_LEN, freq=step)

            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=pred_index)

            predictions_df_all_households[household] = y_pred

            y_true = df.reindex(pred_index)[household].values

            mask = ~np.isnan(y_true) & ~np.isnan(y_pred)

            if mask.sum() == 0:
                print(f"      Warning: no valid truth values for {household} at {day}")
                continue

            rmse = root_mean_squared_error(y_true[mask], y_pred[mask])
            rmse_households.append(rmse)

        if len(rmse_households) == 0:
            avg_rmse_households = np.nan
        else:
            avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        output_path = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimerUnivar_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        if predictions_df_all_households is not None:
            predictions_df_all_households.to_csv(output_path, index=True)
            print("      Saved:", output_path)
        else:
            print("      No predictions generated for this split.")

    # ========================================================
    # COUNTRY RUNTIME
    # ========================================================
    country_runtime = time.time() - country_start_time

    print(
        f"\nTotal runtime for {country}: "
        f"{country_runtime:.2f} seconds"
    )

    # ========================================================
    # SAVE / UPDATE JSON
    # ========================================================
    json_path = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\time_spend_{country}.json"

    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            runtime_dict = json.load(f)
    else:
        runtime_dict = {}

    if "Foundational" not in runtime_dict:
        runtime_dict["Foundational"] = {}

    runtime_dict["Foundational"]["Timer"] = country_runtime

    with open(json_path, "w") as f:
        json.dump(runtime_dict, f, indent=4)

    print(f"Saved/updated runtime JSON: {json_path}")